
# Simple Network by hand

Here, we manually implement a simple network architecture

We want a network with one hidden layer. As activiation in the hidden layer $\sigma$ we apply element-wise ReLu, while no activation is used for the output layer. The forward pass of the network then reads:
$$\hat{y}=\mathbf{W}^{\prime} \sigma(\mathbf{W} \vec{x}+\vec{b})+b^{\prime}$$

For the regression problem the objective function is the mean squared error between the prediction and the true label $y$:
$$
L=(\hat{y}-y)^{2}
$$

Taking the partial derivatives - and diligently the applying chain rule - with respect to the different objects yields:
\begin{aligned}
\frac{\partial L}{\partial b^{\prime}} &=2(\hat{y}-y) \rightarrow \text{derivative wrt the bias of the output layer}\\
\frac{\partial L}{\partial b_{k}} &=2(\hat{y}-y) \mathbf{W}_{k}^{\prime} \Theta\left(\sum_{i} \mathbf{W}_{i k} x_{i}+b_{k}\right) \rightarrow \text{derivative wrt the weight of the output layer}\\
\frac{\partial L}{\partial \mathbf{W}_{k}^{\prime}} &=2(\hat{y}-y) \sigma\left(\sum_{i} \mathbf{W}_{i k} x_{i}+b_{k}\right) \rightarrow \text{derivative wrt the bias of the hidden layer}\\
\frac{\partial L}{\partial \mathbf{W}_{k m}} &=2(\hat{y}-y) \mathbf{W}_{m}^{\prime} \Theta\left(\sum_{i} \mathbf{W}_{i k} x_{i}+b_{m}\right) x_{k} \rightarrow \text{derivative wrt the input weight}
\end{aligned}

Here, $\Theta$ denotes the Heaviside step function. Note that the derivative of the ReLu function wrt its input is the Heaviside step function (1 for positive inputs, 0 for negative input).

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gspread
import pandas as pd
from google.colab import auth
import google.auth

In [29]:
drive.mount('/content/drive')

auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

url_data = 'https://docs.google.com/spreadsheets/d/1S4uwR9xCAQvvUm4wKU9NBCiqcGO9QAuDUGJWWJh7GmA/edit?usp=sharing'
worksheet_data = gc.open_by_url(url_data).sheet1
read = worksheet_data.get_all_values()
data = pd.DataFrame(read[1:], columns=read[0])
#convert to numbers
data = data.apply(pd.to_numeric)

display(data.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ValueError: Unable to parse string "26.05.00" at position 3619

In [ ]:
# Prepare for proper training

data = data.to_numpy() # convert to numpy array
np.random.shuffle(data) # randomly sort examples

# take the first 3000 examples for training
# (remember array slicing from last week)
X_train = data[:3000,:11] # all features except last column
y_train = data[:3000,11]  # quality column

# and the remaining examples for testing
X_test = data[3000:,:11] # all features except last column
y_test = data[3000:,11] # quality column

print("First example:")
print("Features:", X_train[0])
print("Quality:", y_train[0])

# Solutions

The goal is to implement the training of a neural network with one input layer, one hidden layer, and one output layer using gradient descent. We first (below) define the matrices and initialise with random values. We need W, b, W' and b'. The shapes will be:
  * W: (number of hidden nodes, number of inputs) named `W`
  * b: (number of hidden nodes) named `b`
  * W': (number of hidden nodes) named `Wp`
  * b': (one) named `bp`

Your tasks are:     
   * Implement a forward pass of the network as `dnn` (see below)
   * Implement a function that uses one data point to update the weights using gradient descent. You can follow the `update_weights` skeleton below
   * Now you can use the code below (training loop and evaluation) to train the network for multiple data points and even over several epochs. Try to find a set of hyperparameters (number of nodes in the hidden layer, learning rate, number of training epochs) that gives stable results. What is the best result (as measured by the loss on the training sample) you can get?

In [ ]:
# Initialise weights with suitable random distributions
hidden_nodes = 50 # number of nodes in the hidden layer
n_inputs = 11 # input features in the dataset

W = np.random.randn(hidden_nodes,11)*np.sqrt(2./n_inputs)
b = np.random.randn(hidden_nodes)*np.sqrt(2./n_inputs)
Wp = np.random.randn(hidden_nodes)*np.sqrt(2./hidden_nodes)
bp = np.random.randn((1))

print(W.shape)

In [ ]:
# You can use this implementation of the ReLu activation function
def relu(x):
    return np.maximum(x, 0)

In [ ]:
def dnn(x,W,b,Wp,bp):
    # sum_i W'_ki*Relu(sum_j W_ij*x_j + b_i) + b'_k
    return np.dot(Wp, relu(np.dot(W,x) + b)) + bp

In [ ]:
def update_weights(x,y, W, b, Wp, bp):

    lr = 0.00005

    # Calculate the network output
    phi = dnn(x,W,b,Wp,bp)

    # Use the formulas derived to calculate the gradient for each of W,b,Wp,bp
    delta_bp = 2 * (phi - y)
    delta_Wp = 2 * (phi - y) * relu(np.dot(W,x) + b)
    delta_b  = 2 * (phi - y) * Wp * np.heaviside(np.dot(W,x) + b, 0.5)
    delta_W  = 2 * (phi - y) * np.outer(Wp * np.heaviside(np.dot(W,x) + b, 0.5), x)

    # Update the weights/bias following the rule:  X_new = X_old - learning_rate * gradient
    bp -= lr * delta_bp
    Wp -= lr * delta_Wp
    b  -= lr * delta_b
    W  -= lr * delta_W


    return -1 # no return value needed, you can modify the weights in-place

# Training loop and evaluation below

In [ ]:
# The code below implements the training.
# If you correctly implement  dnn and update_weights above,
# you should not need to change anything below.
# (apart from increasing the number of epochs)

train_losses = []
test_losses = []

# How many epochs to train
n_epochs = 50

# Loop over the epochs
for ep in range(n_epochs):

    # Each epoch is a complete over the training data
    for i in range(X_train.shape[0]):

        # pick one example
        x = X_train[i]
        y = y_train[i]

        # use it to update the weights
        update_weights(x,y,W,b,Wp,bp) # this is a GD technique. the full dataset is flowing in order to update the weights

    # Calculate predictions for the full training and testing sample
    y_pred_train = [dnn(x,W,b,Wp,bp)[0] for x in X_train]
    y_pred = [dnn(x,W,b,Wp,bp)[0] for x in X_test]

    # Calculate aver loss / example over the epoch
    train_loss = sum((y_pred_train-y_train)**2) / y_train.shape[0]
    test_loss = sum((y_pred-y_test)**2) / y_test.shape[0]

    # print some information
    print("Epoch:",ep, "Train Loss:", train_loss, "Test Loss:", test_loss)

    # and store the losses for later use
    train_losses.append(train_loss)
    test_losses.append(test_loss)


# After the training:

# Prepare scatter plot
y_pred = [dnn(x,W,b,Wp,bp)[0] for x in X_test]

print("Best loss:", min(test_losses), "Final loss:", test_losses[-1])

print("Correlation coefficient:", np.corrcoef(y_pred,y_test)[0,1])
plt.scatter(y_pred_train,y_train)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# Prepare and loss over time
plt.plot(train_losses,label="train")
plt.plot(test_losses,label="test")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


# Let's implement the same network in keras

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the Keras model
# Input layer has n_inputs features
# Hidden layer has hidden_nodes neurons with ReLU activation
# Output layer has 1 neuron with linear activation (no activation function)
model = keras.Sequential([
    layers.Input(shape=(n_inputs,)),
    layers.Dense(hidden_nodes, activation='relu', name='hidden_layer'),
    layers.Dense(1, name='output_layer')
])

# Compile the model
# Use 'mse' (Mean Squared Error) as the loss function, matching the manual implementation
# Use Adam optimizer as a common choice for Keras models
model.compile(optimizer='adam', loss='mse')

# Display model summary
model.summary()


In [ ]:
# Train the model
# Use the same number of epochs as the manual training for comparison
# Batch size is typically used in Keras, let's pick a reasonable value
history = model.fit(X_train, y_train, epochs=n_epochs, batch_size=32, validation_data=(X_test, y_test), verbose=0)

# Evaluate the model
keras_train_loss = model.evaluate(X_train, y_train, verbose=0)
keras_test_loss = model.evaluate(X_test, y_test, verbose=0)

print(f"Keras Train Loss: {keras_train_loss}")
print(f"Keras Test Loss: {keras_test_loss}")

# Perform inference
y_pred_keras_train = model.predict(X_train).flatten()
y_pred_keras_test = model.predict(X_test).flatten()

# Calculate correlation coefficient
keras_corr_coeff = np.corrcoef(y_pred_keras_test, y_test)[0,1]
print(f"Keras Correlation coefficient (Test): {keras_corr_coeff}")



In [ ]:
# Plotting results

# Plot training & validation loss values
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model loss (Keras)')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')

# Plot predicted vs true values for test set
plt.subplot(1, 2, 2)
plt.scatter(y_pred_keras_test, y_test)
plt.xlabel("Predicted (Keras)")
plt.ylabel("True")
plt.title('Predicted vs True values (Keras Test Set)')
plt.show()

# Compare the output for the two networks

In [ ]:
print("\n--- Confronto tra le Reti Neurali ---")
print("\n### Rete implementata manualmente ###")
print(f"  Loss su training (finale): {train_losses[-1]:.4f}")
print(f"  Loss su test (migliore): {min(test_losses):.4f}")
print(f"  Coefficiente di correlazione (test): {np.corrcoef(y_pred, y_test)[0,1]:.4f}")

print("\n### Rete implementata con Keras ###")
print(f"  Loss training (finale): {keras_train_loss:.4f}")
print(f"  Loss su test (finale): {keras_test_loss:.4f}")
print(f"  Coefficiente di correlazione (test): {keras_corr_coeff:.4f}")


# Plot comparativo delle perdite nel tempo
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Manual Train Loss')
plt.plot(test_losses, label='Manual Test Loss')
plt.plot(history.history['loss'], label='Keras Train Loss', linestyle='--')
plt.plot(history.history['val_loss'], label='Keras Test Loss', linestyle='--')
plt.title('Confronto delle Perdite nel Tempo')
plt.xlabel('Epoche')
plt.ylabel('Perdita (MSE)')
plt.legend()
plt.grid(True)

# Plot comparativo delle predizioni vs valori veri (Test Set)
plt.subplot(1, 2, 2)
plt.scatter(y_pred, y_test, alpha=0.6, label='Manuale')
plt.scatter(y_pred_keras_test, y_test, alpha=0.6, label='Keras')
plt.xlabel('Predetto')
plt.ylabel('Vero')
plt.title('Confronto Predetto vs Vero (Test Set)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_pred, y_pred_keras_test, alpha=0.6)
plt.xlabel('Prediction from Manual Network')
plt.ylabel('Prediction from Keras Network')
plt.title('Scatter Plot: Manual vs Keras Predictions (Test Set)')

# Add a diagonal line for perfect agreement
min_val = min(min(y_pred), min(y_pred_keras_test))
max_val = max(max(y_pred), max(y_pred_keras_test))
plt.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='Perfect Agreement')

plt.legend()
plt.grid(True)
plt.show()